# Entity Post-Processing Pipeline

This notebook brings together the five steps of the post-processing pipeline for CamemBERT model predictions:

1. **Context Addition** (`add_context`)
2. **Span Extraction** (`extract_spans`)
3. **Entity Normalization** (`normalize_spans`)
4. **Metadata Extraction** (`extract_metadata`)
5. **Span–Metadata Merging** (`join_spans_metadata`)

Each step can be executed independently to inspect intermediate results, or the entire pipeline can be executed sequentially using the **Run Complete Pipeline** section.

## Configurations
Modifiez uniquement ce bloc pour chaque nouveau corpus. 

In [43]:
LEGISLATURE = "XIV"   # XIV | XV | XVI
TYPE_CORPUS = "qg"    # qe | qg (question écrite | question orale)
# Entrées
PREDICTIONS_INPUT        = f"predict/predictions_{TYPE_CORPUS}{LEGISLATURE}.json"
POOL_REF                 = f"pool_{TYPE_CORPUS}_{LEGISLATURE}.json"
DICT_MAIN                = "dictionary/main_dictionary.json"
DICT_PERSON              = "dictionary/person_dictionary.json"
DICT_GENERAL             = "dictionary/general_dictionary.json"
METADATA_INPUTS          = [f"metadata_{TYPE_CORPUS}{LEGISLATURE}.json"]  # liste : accepte XIV+XV+XVI
XML_INPUT_MODE           = "bundle"     # "single" (XVe/XVIe) ou "bundle" (XIVe)
XML_INPUT_PATH           = f"/Users/lihuizhu/Desktop/corpus/Questions_gouvernement_XIV.xml"  # gouvermentales

# Sorties
PREDICTIONS_WITH_CONTEXT = f"predict/predictions_{TYPE_CORPUS}{LEGISLATURE}_context.json"
SPANS_OUTPUT             = f"predict/spans_{TYPE_CORPUS}{LEGISLATURE}.json"
SPANS_NORMALIZED         = f"predict/spans_{TYPE_CORPUS}{LEGISLATURE}_normalized.json"
METADATA_OUTPUT          = f"metadata_{TYPE_CORPUS}{LEGISLATURE}.json"
FINAL_CSV                = f"predict/sortie_{TYPE_CORPUS}{LEGISLATURE}.csv"

MAX_SKIP = 5   # phrases ponctuelles à sauter pour trouver un vrai contexte prev/next


## Imports

In [ ]:
import json, csv, glob, re, os, unicodedata

from pathlib import Path
from collections import Counter, defaultdict
from xml.etree import ElementTree as ET

import pandas as pd

## Fonctions partagées

In [21]:
def read_jsonl(path: str):
    """Lit un fichier JSON array."""
    with open(path, encoding="utf-8") as f:
        content = f.read().strip()
    if not content:
        return
    if content.startswith("["):
        yield from json.loads(content)
    else:
        for line in content.splitlines():
            line = line.strip()
            if line:
                yield json.loads(line)

def write_jsonl(path: str, items):
    """Écrit une séquence de dicts en JSON."""
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Normalisation de base 
def normalize_entity(text: str) -> str:
    """Nettoie le texte brut du span (espaces, ponctuation, balises résiduelles)."""
    text = unicodedata.normalize("NFKC", text)
    text = text.strip()
    text = re.sub(r"</?em>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s*'\s*", "'", text)
    text = re.sub(r"([ldjtmcsnLDJTMCSN])'\s+", r"\1'", text)
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"[,;.!?]+$", "", text).strip()
    return text

# Whitelists
GEO_WHITELIST = re.compile(
    r'\b(grand est|sud est|nord est|sud ouest|nord ouest|'
    r'île de france|pays de la loire|hauts de france|'
    r'provence|occitanie|bretagne|normandie|bourgogne|'
    r'centre val de loire|nouvelle aquitaine|auvergne)$',
    re.I
)
MEDIA_WHITELIST = {
    'le figaro', 'le figaro magazine', 'le monde', 'le monde diplomatique',
    'le parisien', 'le point', 'le canard enchâîné', 'le canard',
    'le journal du dimanche', 'le jdd', 'le nouvel observateur', 'le nouvel obs',
    'le nouvel économiste', 'le quotidien du médecin', 'le quotidien du pharmacien',
    'le télégramme', 'le dauphiné libéré', 'le progrès', 'le républicain lorrain',
    'le bien public', "le courrier de l'ouest", 'le maine libre',
    'le populaire du centre', 'le média', 'la croix', 'la dépêche du midi',
    'la dépêche', 'la provence', 'la montagne', 'la tribune',
    'la lettre du cadre', 'la recherche', 'la chaîne parlementaire',
    'les échos', 'les inrockuptibles', 'les inrocks', 'les jours',
}
ASSO_WHITELIST = {
    'les restos du cœur', 'les restaurants du cœur', 'les petits frères des pauvres',
    'les amis de la terre', 'les amis de la terre - france', 'les amis de sam',
    'les morts de la rue', 'la quadrature du net', "la ligue de l'enseignement",
    "la ligue des droits de l'homme", 'la fondation droit animal, éthique et sciences',
    "la mutuelle des étudiants", 'la banque postale', 'la poste',
    'la coopération agricole', 'la france insoumise', 'le secours catholique',
    'le secours populaire', 'le mouvement hlm', 'les semaines sociales de france',
    'la caisse des dépôts', 'les hospices civils de lyon',
    'la réunion des musées nationaux', 'la documentation française',
}

def is_whitelisted(text: str) -> bool:
    tl = text.lower().strip()
    return tl in MEDIA_WHITELIST or tl in ASSO_WHITELIST or bool(GEO_WHITELIST.search(tl))

def is_whitelisted(text: str) -> bool:
    """Retourne True si le texte ne doit PAS être modifié par strip_borders."""
    tl = text.lower().strip()
    if tl in MEDIA_WHITELIST or tl in ASSO_WHITELIST:
        return True
    if GEO_WHITELIST.search(tl):
        return True
    return False
# strip_borders

VERB_SUFFIX = re.compile(
    r',?\s+(a|est|sont|ont|peut|doit|fait|dit|semble|indique|précise|note|'
    r'rappelle|souligne|estime|considère|propose|suggère|recommande|demande|'
    r'appelle|invite|encourage|souhaite|entend|vise|tend|cherche|tente|prévoit|'
    r'dispose|comprend|inclut|contient|regroupe|rassemble|réunit|associe|'
    r'mêle|lie|relie|rattache|appartient|dépend|relève|ressort|découle|'
    r'résulte|provient|vient|arrive|revient|reste|demeure|devient|'
    r"apparaît|s'avère|affirme|déclare|annonce|publie|constate|observe|"
    r"signale|alerte|s'inquiète|s'engage|s'oppose|vote|adopte|"
    r'rejette|approuve|refuse|accepte|décide|choisit|préfère|'
    r'intervient|participe|contribue|répond|réagit|précise|confirme)$',
    re.I
)
CONJ_SUFFIX = re.compile(
    r'\s+(qui|ainsi|que|au|aux|notamment|comme|et|ou|dont|par|sur|dans|'
    r'pour|avec|car|mais|donc|or|ni|soit|en|à|'
    r"de la|du|de l'|des|à la|à l'|"
    r'sur le|sur la|sur les|dans le|dans la|dans les|'
    r'pour le|pour la|pour les)$',
    re.I
)
BOTH_SIDES = {
    "de", "du", "d'", "d’", "des", "et", "ou", "par", "pour", "avec",
    "dans", "en", "sur", "ce", "cet", "cette", "ces",
    ",", ";", ".", "!", "?", "(", ")", "-", "–", "—", "«", "»",
}
RIGHT_ONLY    = {"le", "la", "les", "l'", "l’"}
LEFT_ARTICLES = re.compile(r"^(le|la|les|l'|l’|un|une|des|du|de|d'|d’)\s+", re.I)
PUNCT_RESIDUAL = re.compile(r'^[,;:«»\s]+|[,;:«»\s]+$')

def strip_borders(text: str, n_passes: int = 3) -> str:
    """Retire les mots fonctionnels parasites en bordure de span (whitelists respectées)."""
    if is_whitelisted(text):
        return text
    original = text
    for _ in range(3):
        if GEO_WHITELIST.search(text.lower()):
            break
        m = VERB_SUFFIX.search(text)
        if m:
            text = text[:m.start()].strip()
        else:
            break
    for _ in range(2):
        m = CONJ_SUFFIX.search(text)
        if m:
            text = text[:m.start()].strip()
        else:
            break
    for _ in range(n_passes):
        changed = False
        for token in BOTH_SIDES:
            if text.lower().startswith(token + " ") or text.lower() == token:
                text = text[len(token):].strip(); changed = True; break
        for token in BOTH_SIDES | RIGHT_ONLY:
            if text.lower().endswith(" " + token) or text.lower() == token:
                text = text[:-len(token)].strip(); changed = True; break
        if not changed:
            break
    if not is_whitelisted(text):
        m = LEFT_ARTICLES.match(text)
        if m:
            rest = text[m.end():].strip()
            if len(rest) >= 4:
                text = rest
    text = PUNCT_RESIDUAL.sub('', text).strip()
    return text if text else original

def strip_borders(text: str, n_passes: int = 3) -> str:
    """
    Version améliorée de strip_borders avec whitelist médias/associations.
    Si le texte est whitelisted, retourné tel quel.
    """
    if is_whitelisted(text):
        return text

    original = text

    # 1. Suffixe verbal (itératif, protège les noms géographiques)
    for _ in range(3):
        if GEO_WHITELIST.search(text.lower()):
            break
        m = VERB_SUFFIX.search(text)
        if m:
            text = text[:m.start()].strip()
        else:
            break

    # 2. Suffixe conjonction/préposition
    for _ in range(2):
        m = CONJ_SUFFIX.search(text)
        if m:
            text = text[:m.start()].strip()
        else:
            break

    # 3. Préfixes/suffixes fonctionnels (n passes)
    for _ in range(n_passes):
        changed = False
        for token in BOTH_SIDES:
            if text.lower().startswith(token + " ") or text.lower() == token:
                text = text[len(token):].strip()
                changed = True
                break
        for token in BOTH_SIDES | RIGHT_ONLY:
            if text.lower().endswith(" " + token) or text.lower() == token:
                text = text[:-len(token)].strip()
                changed = True
                break
        if not changed:
            break

    # 4. Article en tête (seulement si résidu >= 4 chars et pas whitelisted)
    if not is_whitelisted(text):
        m = LEFT_ARTICLES.match(text)
        if m:
            rest = text[m.end():].strip()
            if len(rest) >= 4:
                text = rest

    # 5. Ponctuation résiduelle
    text = PUNCT_RESIDUAL.sub('', text).strip()

    return text if text else original


## Étape 1 — `add_context`
Enrichit les résultats avec les champs `prev` / `next` :

- **Entrée** : `PREDICTIONS_INPUT` (JSON- sortie brute du modèle)
- **Référence** : `POOL_REF` (JSON - toutes les phrases segmentées du corpus)
- **Sortie** : `PREDICTIONS_WITH_CONTEXT`


In [22]:
punct_re = re.compile(r"^[\W_]+$", re.UNICODE)

# ── Helpers ───────────────────────────────────────────────────────────────────

def punct_only(sent: str) -> bool:
    """True si la phrase ne contient que de la ponctuation / espaces."""
    if sent is None:
        return True
    s = str(sent).strip()
    if not s:
        return True
    s = re.sub(r"\s+", "", s)
    return bool(punct_re.fullmatch(s))


def extract_num(sent_id: str) -> int | None:
    """
    Extrait le numéro de phrase depuis l'id.
    Ex. "QANR5L16QE916S03" -> 3
    Retourne None si le pattern n'est pas trouvé.
    """
    # m = re.search(r"S(\d+)$", str(sent_id))
    m = re.search(r"Q(\d+)$", str(sent_id))
    return int(m.group(1)) if m else None

def build_index(ref_path: str) -> dict:
    """
    Lit le fichier JSON de référence et construit
    {(uid, num): text, ...}
    """
    index = {}
    with open(ref_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            uid     = str(item.get("uid", ""))
            sent_id = item.get("id", "")
            text    = item.get("text", "")
            num     = extract_num(sent_id)
            if uid and num is not None:
                index[(uid, num)] = text
    print(f"{len(index)} phrases indexées.")
    return index

def find_context(uid: str, num: int, uid_to_sent: dict,
                 direction: int, max_skip: int = MAX_SKIP) -> str | None:
    """
    Cherche la phrase voisine non-ponctuation dans la direction donnée.
    direction = -1 -> phrase précédente
    direction = +1 -> phrase suivante
    """
    pos = num + direction
    steps = 0
    while steps < max_skip:
        sent = uid_to_sent.get((uid, pos))
        if sent is None:
            return None                   # début/fin du text
        if not punct_only(sent):
            return sent
        pos += direction
        steps += 1
    return None

def add_context(input_path: str, ref_path: str, output_path: str) -> None:
    index = build_index(ref_path)

    # Lecture du JSON array de prédictions
    with open(input_path, encoding="utf-8") as f:
        content = f.read().strip()
        if content.startswith('['):
            predictions = json.loads(content)
        else:
            predictions = [json.loads(line) for line in content.split('\n') if line.strip()]

    unk = 0
    for item in predictions:
        uid     = str(item.get("uid", ""))
        sent_id = item.get("id", "")
        num     = extract_num(sent_id)

        if not uid or num is None:
            item["prev"] = None
            item["next"] = None
            unk += 1
            continue

        item["prev"] = find_context(uid, num, index, direction=-1)
        item["next"] = find_context(uid, num, index, direction=+1)

    # Sauvegarde
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    print(f"{len(predictions)} entrées écrites dans : {output_path}")
    print(f"{unk} entrée(s) sans uid/id valide.")

# Exécution
add_context(PREDICTIONS_INPUT, POOL_REF, PREDICTIONS_WITH_CONTEXT)


64409 phrases indexées.
64409 entrées écrites dans : predict/predictions_qgXV_context.json
0 entrée(s) sans uid/id valide.


## Étape 2 — `extract_spans`
Parcourt les séquences IOB et extrait une ligne JSONL par occurrence .

- **Entrée** : `PREDICTIONS_WITH_CONTEXT`
- **Sortie** : `SPANS_OUTPUT`

In [23]:
def detokenize(tokens):
    """
    Recolle une liste de tokens en texte lisible.
    Règle simple : pas d'espace avant la ponctuation ni après une apostrophe/tiret,
    pas d'espace après une apostrophe.
    """
    text = ""
    no_space_before = {",", ".", ";", ":", "!", "?", ")", "»", "'", "’"}
    no_space_after = {"(", "«", "'", "’", "-"}
    for i, tok in enumerate(tokens):
        if i == 0:
            text = tok
            continue
        prev = tokens[i - 1]
        if tok in no_space_before or prev in no_space_after:
            text += tok
        else:
            text += " " + tok
    return text


def extract_spans_from_labels(tokens, labels):
    """
    Parcourt une séquence IOB et retourne la liste des spans.
    Tolérant aux erreurs de tagging (I-SRC sans B-SRC précédent -> traité comme un début de span).
    """
    spans = []
    start = None
    for i, lab in enumerate(labels):
        if lab == "B-SRC":
            if start is not None:
                spans.append((start, i - 1))
            start = i
        elif lab == "I-SRC":
            if start is None:
                # I-SRC orphelin : on démarre quand même un span ici
                start = i
        else:  # "O" ou autre
            if start is not None:
                spans.append((start, i - 1))
                start = None
    if start is not None:
        spans.append((start, len(labels) - 1))
    return spans

def extract_spans(input_path, output_path, csv_output=None):
    n_sentences = 0
    n_spans = 0
    skipped = 0

    with open(output_path, "w", encoding="utf-8") as out:
        for rec in read_jsonl(input_path):
            n_sentences += 1
            tokens = rec.get("tokens")
            labels = rec.get("labels") or rec.get("pred_labels")

            if not tokens or not labels:
                skipped += 1
                continue
            if len(tokens) != len(labels):
                skipped += 1
                continue

            spans = extract_spans_from_labels(tokens, labels)
            for (start, end) in spans:
                span_tokens = tokens[start:end + 1]
                occurrence = detokenize(span_tokens)
                out.write(json.dumps({
                    "uid": rec.get("uid"),
                    "id": rec.get("id"),
                    "text": rec.get("text"),
                    "prev": rec.get("prev"),
                    "next": rec.get("next"),
                    "span_start": start,
                    "span_end": end,
                    "occurrence": occurrence,
                }, ensure_ascii=False) + "\n")
                n_spans += 1

    print(f"Phrases lues       : {n_sentences}")
    print(f"Phrases ignorées   : {skipped} (tokens/labels manquants ou désalignés)")
    print(f"Occurrences extraites : {n_spans}")
    print(f"-> {output_path}")

extract_spans(PREDICTIONS_WITH_CONTEXT, SPANS_OUTPUT)


Phrases lues       : 64409
Phrases ignorées   : 0 (tokens/labels manquants ou désalignés)
Occurrences extraites : 9106
-> predict/spans_qgXV.json


## Étape 3 — `normalize_spans`  
Sur chaque span :

1. `normalize_entity` +`strip_borders`  =>  `entity_clean`
2. Lookup séquentiel exclusif **ORG/PER/GEN**  => `entity_normalized` + `entity_type`  
   (pas de match => champs vides)

- **Entrée** : `SPANS_OUTPUT`
- **Sortie** : `SPANS_NORMALIZED`


In [25]:
STOP_WORDS = {
    "de", "du", "des", "d", "la", "le", "les", "l",
    "en", "au", "aux", "et", "ou", "pour", "sur", "par",
    "un", "une", "à", "a",
}

def load_dict(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def build_flat_lookup(d):
    """
    Aplati un dictionnaire {key: {canonical, variants}} en {surface_lower: canonical},
    en indexant à la fois la clé et CHAQUE variant — sinon les variantes orthographiques
    (ex: "Cour des compte" sans s, alors que la clé est "cour des comptes") sont ratées.
    """
    flat = {}
    for key, val in d.items():
        canonical = val["canonical"]
        flat[key.lower()] = canonical
        for variant in val.get("variants", []):
            flat[variant.lower()] = canonical
    return flat


def comparison_key(s):
    """Clé de secours : ponctuation -> espace, mots-outils retirés aux bords seulement."""
    tokens = re.sub(r"[^\w\s'-]", " ", s).split()
    while tokens and tokens[0].lower().strip("'") in STOP_WORDS:
        tokens.pop(0)
    while tokens and tokens[-1].lower().strip("'") in STOP_WORDS:
        tokens.pop()
    return " ".join(t.lower() for t in tokens)


def normalize_spans(input_path, main_path, person_path, general_path, output_path):
    """
    Applique les 3 dictionnaires en 3 passes successives et exclusives, par
    ordre de priorité : main (institutions/orgs, le plus spécifique) ->
    person (noms propres, peu d'ambiguïté) -> general (termes génériques,
    le plus large, donc en dernier rempart).
    Dès qu'un span trouve un match, les étapes suivantes ne le touchent plus.
    Pas de match dans aucun des 3 -> entity_normalized = "" (champ vide).

    Chaque dictionnaire passé ici est déjà une lookup PLATE {surface_lower: canonical}
    construite par build_flat_lookup() — donc on matche aussi bien sur la clé que
    sur n'importe quel variant orthographique recensé.
    """
    main_dict = build_flat_lookup(load_dict(main_path))
    person_dict = build_flat_lookup(load_dict(person_path))
    general_dict = build_flat_lookup(load_dict(general_path))
    print(f"Dictionnaires — ORG:{len(main_dict)} | PER:{len(person_dict)} | GEN:{len(general_dict)} surfaces")


    ordered_dicts = [("ORG", main_dict), ("PER", person_dict), ("GEN", general_dict)]
    stats = Counter({"ORG": 0, "PER": 0, "GEN": 0, "none": 0, "total": 0})
    results = []
    for span in read_jsonl(input_path):
        stats["total"] += 1

        entity_clean = strip_borders(normalize_entity(span.get("occurrence", "")))
        span["entity_clean"] = entity_clean

        raw = span["entity_clean"]
        key = raw.lower()
        ck = None  # calculé seulement si nécessaire
    
        for tag, flat in ordered_dicts:
            if key in flat:
                canonical = flat[key]
            else:
                if ck is None:
                    ck = comparison_key(raw)
                if ck in flat:
                    canonical = flat[ck]
                else:
                    continue
            span["entity_normalized"] = canonical
            span["entity_type"] = tag
            stats[tag] += 1
            break
        else:
            span["entity_normalized"] = ""
            span["entity_type"] = ""
            stats["none"] += 1

        results.append(span)

    write_jsonl(output_path, results)

    total = stats["total"]
    print(f"\nRésultat normalisation ({total} spans) :")
    for k in ("ORG", "PER", "GEN"):
        print(f"  trouvé via {k:3s}: {stats[k]} ({stats[k]/total*100:.1f}%)")
    print(f"  non trouvé (vide) : {stats['none']} ({stats['none']/total*100:.1f}%)")
    print(f" Entregistré -> {output_path}")

normalize_spans(SPANS_OUTPUT, DICT_MAIN, DICT_PERSON, DICT_GENERAL, SPANS_NORMALIZED)

Dictionnaires — ORG:3336 | PER:363 | GEN:327 surfaces

Résultat normalisation (9106 spans) :
  trouvé via ORG: 4387 (48.2%)
  trouvé via PER: 298 (3.3%)
  trouvé via GEN: 1097 (12.0%)
  non trouvé (vide) : 3324 (36.5%)
 Entregistré -> predict/spans_qgXV_normalized.json


## Étape 4 — `extract_metadata`  
*(flux parallèle, indépendant des étapes 1–3)*

Extrait les métadonnées XML de l'Assemblée nationale.

- **`single`** (XVe, XVIe) : un `.xml` par question → passer un dossier
- **`bundle`** (XIVe) : un seul gros `.xml` → passer le chemin du fichier

- **Sortie** : `METADATA_OUTPUT`

In [35]:
import glob
from xml.etree import ElementTree as ET

NS = {"an": "http://schemas.assemblee-nationale.fr/referentiel"}


def detect_ns_prefix(root):
    """
    Détecte si la racine utilise le namespace AN (XVe/XVIe) ou pas (XIVe).
    Retourne "an:" si le namespace est présent, "" sinon.
    ElementTree représente un tag avec namespace comme '{uri}tag'.
    """
    if root.tag.startswith("{"):
        return "an:"
    return ""


def _text(el, path_no_ns, ns_prefix):
    """
    Retourne le texte d'un sous-élément, en préfixant chaque segment du
    chemin avec ns_prefix ("an:" ou "") selon le fichier source.
    path_no_ns : chemin écrit SANS préfixe, ex: "identifiant/legislature"
    """
    if ns_prefix:
        path = "/".join(f"{ns_prefix}{seg}" for seg in path_no_ns.split("/"))
        found = el.find(path, NS)
    else:
        found = el.find(path_no_ns)
    if found is None:
        return None
    return found.text.strip() if found.text else None


def _findall(el, path_no_ns, ns_prefix):
    if ns_prefix:
        path = "/".join(f"{ns_prefix}{seg}" for seg in path_no_ns.split("/"))
        return el.findall(path, NS)
    return el.findall(path_no_ns)


def extract_analyse(question_el, ns_prefix):
    """
    Extrait le texte de l'analyse (mots-clés indexationAN).

    Deux variantes observées selon les fichiers source :
      - "analyses/analyse" (pluriel, minuscules)   -> ex. XIVe/XVIe
      - "ANALYSE/ANA"       (singulier, majuscules) -> ex. XVe

    On essaie les deux chemins et on joint tous les textes trouvés avec " | "
    (au cas où il y aurait plusieurs <ANALYSE> ou plusieurs <analyse>).
    """
    analyses = _findall(question_el, "indexationAN/analyses/analyse", ns_prefix)
    if not analyses:
        analyses = _findall(question_el, "indexationAN/ANALYSE/ANA", ns_prefix)
    return " | ".join(a.text.strip() for a in analyses if a.text) or None

def _parse_question(q, ns=None):
    if ns is None: ns = detect_ns_prefix(q)
    return {
        "uid":              _text(q, "uid", ns),
        "legislature":      _text(q, "identifiant/legislature", ns),
        "type":             _text(q, "type", ns),
        "rubrique":         _text(q, "indexationAN/rubrique", ns),
        "analyse":          extract_analyse(q, ns),
        "acteurRef":        _text(q, "auteur/identite/acteurRef", ns),
        "parti_abrege":     _text(q, "auteur/groupe/abrege", ns),
        "ministere_abrege": _text(q, "minInt/abrege", ns),
        "dateJO":           _text(q, "textesReponse/texteReponse/infoJO/dateJO", ns),  # textesQuestion/texteQuestion/infoJO/dateJO
        "libelleCloture":   _text(q, "cloture/libelleCloture", ns),
    }

def extract_metadata(mode, input_path, output_path):
    def _iter():
        if mode == "single":
            for path in sorted(glob.glob(os.path.join(input_path, "*.xml"))):
                try: yield _parse_question(ET.parse(path).getroot())
                except ET.ParseError as e: print(f"[WARN] XML ignoré : {path} ({e})")
        elif mode == "bundle":
            root = ET.parse(input_path).getroot()
            ns   = detect_ns_prefix(root)
            qs   = _findall(root, "question", ns) or root.findall(".//question")
            print(f"[bundle] {len(qs)} <question> trouvés")
            for q in qs: yield _parse_question(q, ns)
        else: raise ValueError("mode doit être 'single' ou 'bundle'")

    n = 0
    with open(output_path, "w", encoding="utf-8") as f:
        for rec in _iter():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); n += 1
    print(f"{n} questions extraites → {output_path}")

extract_metadata(XML_INPUT_MODE, XML_INPUT_PATH, METADATA_OUTPUT)


1940 questions extraites → metadata_qgXVI.json


## Étape 5 — `join_spans_metadata`
Associe chaque span normalisé aux métadonnées de sa question (clé `uid`).  
Produit le CSV final aligné sur les **18 colonnes** du modèle.

- **Entrée** : `SPANS_NORMALIZED` + `METADATA_INPUTS`
- **Sortie** : `FINAL_CSV` 


In [34]:
import csv

def build_metadata_index(metadata_paths):
    """Construit un dict uid -> métadonnées, à partir d'un ou plusieurs fichiers."""
    index = {}
    n_total = 0
    n_dupes = 0
    for path in metadata_paths:
        for rec in read_jsonl(path):
            uid = rec.get("uid")
            if uid is None:
                continue
            n_total += 1
            if uid in index:
                n_dupes += 1
            index[uid] = rec
    print(f"Metadata indexée : {len(index)} uid uniques (sur {n_total} lignes lues, {n_dupes} doublons écrasés)")
    return index


FIELDNAMES = [
    "identifiant question", "identifiant phrase", "phrase",
    "phrase précédente", "phrase suivante",
    "occurrence identifiée", "entité nettoyée", "entité normalisée", "entity_type",
    "identifiant question [uid]", "législature [legislature]", "type [type]",
    "rubrique [rubrique]", "synthèse [analyse]", "député [acteurRef]",
    "parti [abrege dans groupe]", "ministère interpellé [abrege sous organeRef]",
    "date [dateJO]", "mode de cloture [libellecloture]",
]


def join_spans_metadata(spans_path, metadata_paths, output_path):
    metadata_index = build_metadata_index(metadata_paths)

    n_rows = 0
    n_no_metadata = 0
    missing_uids = set()

    with open(output_path, "w", encoding="utf-8", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=FIELDNAMES)
        writer.writeheader()

        for span in read_jsonl(spans_path):
            uid = span.get("uid")
            meta = metadata_index.get(uid)

            if meta is None:
                n_no_metadata += 1
                missing_uids.add(uid)
                meta = {}  # on écrit la ligne quand même, colonnes metadata vides

            writer.writerow({
                "identifiant question": span.get("uid"),
                "identifiant phrase": span.get("id"),
                "phrase": span.get("text"),
                "phrase précédente": span.get("prev"),
                "phrase suivante": span.get("next"),
                "occurrence identifiée": span.get("occurrence"),
                "entité nettoyée": span.get("entity_clean", ""),
                "entité normalisée": span.get("entity_normalized", ""),
                "entity_type": span.get("entity_type", ""),
                "identifiant question [uid]": meta.get("uid"),
                "législature [legislature]": meta.get("legislature"),
                "type [type]": meta.get("type"),
                "rubrique [rubrique]": meta.get("rubrique"),
                "synthèse [analyse]": meta.get("analyse"),
                "député [acteurRef]": meta.get("acteurRef"),
                "parti [abrege dans groupe]": meta.get("parti_abrege"),
                "ministère interpellé [abrege sous organeRef]": meta.get("ministere_abrege"),
                "date [dateJO]": meta.get("dateJO"),
                "mode de cloture [libellecloture]": meta.get("libelleCloture"),
            })
            n_rows += 1

    print(f"\n{n_rows} lignes écrites -> {output_path}")
    print(f"Occurrences sans métadonnée trouvée (uid absent de l'index) : {n_no_metadata}")
    if missing_uids:
        sample = list(missing_uids)[:10]
        print(f"  uid manquants (échantillon) : {sample}{'...' if len(missing_uids) > 10 else ''}")

join_spans_metadata(SPANS_NORMALIZED, METADATA_INPUTS, FINAL_CSV)

Metadata indexée : 1940 uid uniques (sur 1940 lignes lues, 0 doublons écrasés)

3323 lignes écrites -> predict/sortie_qgXVI.csv
Occurrences sans métadonnée trouvée (uid absent de l'index) : 0


## Run complet
Lance les 5 étapes en séquence.  
Prérequis : cellules *Configurations*, *Imports*, *Fonctions partagées* et 5 étapes déjà exécutées.
Remarque : les appels de fonctions à la fin des cellules des cinq étapes peuvent être mis en commentaire afin d'éviter une double exécution. Le Run complet permet alors d'exécuter l'ensemble du pipeline en une seule fois.

Pour vérifier le résultat d'une étape spécifique, il suffit de décommenter l'appel de fonction correspondant à la fin de sa cellule et de l'exécuter individuellement, à condition que ses fichiers d'entrée soient disponibles.

In [44]:
print("=" * 60)
print(f"Pipeline DAPEPANe — législature {LEGISLATURE}")
print("=" * 60)

print("\n[1/5] add_context")
add_context(PREDICTIONS_INPUT, POOL_REF, PREDICTIONS_WITH_CONTEXT)

print("\n[2/5] extract_spans")
extract_spans(PREDICTIONS_WITH_CONTEXT, SPANS_OUTPUT)

print("\n[3/5] normalize_spans")
normalize_spans(SPANS_OUTPUT, DICT_MAIN, DICT_PERSON, DICT_GENERAL, SPANS_NORMALIZED)

print("\n[4/5] extract_metadata")
extract_metadata(XML_INPUT_MODE, XML_INPUT_PATH, METADATA_OUTPUT)

print("\n[5/5] join_spans_metadata")
join_spans_metadata(SPANS_NORMALIZED, METADATA_INPUTS, FINAL_CSV)

print("\n✓ Pipeline terminé.")


Pipeline DAPEPANe — législature XIV

[1/5] add_context
61593 phrases indexées.
61593 entrées écrites dans : predict/predictions_qgXIV_context.json
0 entrée(s) sans uid/id valide.

[2/5] extract_spans
Phrases lues       : 61593
Phrases ignorées   : 0 (tokens/labels manquants ou désalignés)
Occurrences extraites : 9143
-> predict/spans_qgXIV.json

[3/5] normalize_spans
Dictionnaires — ORG:3336 | PER:363 | GEN:327 surfaces

Résultat normalisation (9143 spans) :
  trouvé via ORG: 4743 (51.9%)
  trouvé via PER: 305 (3.3%)
  trouvé via GEN: 1098 (12.0%)
  non trouvé (vide) : 2997 (32.8%)
 Entregistré -> predict/spans_qgXIV_normalized.json

[4/5] extract_metadata
[bundle] 4717 <question> trouvés
4717 questions extraites → metadata_qgXIV.json

[5/5] join_spans_metadata
Metadata indexée : 4717 uid uniques (sur 4717 lignes lues, 0 doublons écrasés)

9143 lignes écrites -> predict/sortie_qgXIV.csv
Occurrences sans métadonnée trouvée (uid absent de l'index) : 0

✓ Pipeline terminé.
